In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

SINOGRAMS_ZIP = "/content/drive/MyDrive/sinograms.zip"
FBP_ZIP = "/content/drive/MyDrive/fbp.zip"

LOCAL_SINOGRAMS_DIR = Path("/content/sinograms")
LOCAL_FBP_DIR = Path("/content/fbp")

if not LOCAL_SINOGRAMS_DIR.exists():
    print("Estraggo i sinogrammi...")
    get_ipython().system('unzip -q "{SINOGRAMS_ZIP}" -d /content/')
else:
    print("Sinogrammi gia' estratti, salto.")

if not LOCAL_FBP_DIR.exists():
    print("Estraggo FBP...")
    get_ipython().system('unzip -q "{FBP_ZIP}" -d /content/')
else:
    print("FBP gia' estratto, salto.")

Mounted at /content/drive
Estraggo i sinogrammi...
Estraggo FBP...


In [2]:
# ============================================================
# Installazione IPPy + patch del bug CuPy (stessa patch gia' usata in 03_TV.ipynb)
# ============================================================
get_ipython().system('pip install -q git+https://github.com/devangelista2/IPPy.git')
try:
    get_ipython().system('pip install -q cupy-cuda12x')
    import cupy  # noqa: F401
    print("CuPy installato correttamente.")
except Exception as e:
    print(f"CuPy non disponibile ({e}).")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 6.1 MB/s eta 0:00:00
CuPy installato correttamente.


In [3]:
import re, glob

candidates = glob.glob("/usr/**/dist-packages/IPPy/operators.py", recursive=True) + \
             glob.glob("/usr/**/site-packages/IPPy/operators.py", recursive=True)
assert candidates, "Non trovo operators.py di IPPy installato."
operators_path = candidates[0]

with open(operators_path) as f:
    src = f.read()

pattern = re.compile(
    r'(elif torch\.cuda\.is_available\(\) and not force_cpu:\s*\n'
    r'\s*warnings\.warn\(\s*\n'
    r'\s*"CUDA available but CuPy not found\. CTProjector limited to CPU operations for ASTRA data transfer\."\s*\n'
    r'\s*\)\s*\n'
    r'\s*# Force CPU mode if CuPy isn\'t there for GPU data handling\s*\n'
    r'(\s*)self\.use_gpu = False)'
)

def _fix(m):
    indent = m.group(2)
    return (
        "elif torch.cuda.is_available() and not force_cpu:\n"
        f"{indent}try:\n"
        f"{indent}    import cupy  # noqa: F401\n"
        f"{indent}except ImportError:\n"
        f"{indent}    warnings.warn(\n"
        f'{indent}        "CUDA available but CuPy not found. CTProjector limited to CPU operations for ASTRA data transfer."\n'
        f"{indent}    )\n"
        f"{indent}    self.use_gpu = False\n"
    )

new_src, n = pattern.subn(_fix, src)
if n == 1:
    with open(operators_path, "w") as f:
        f.write(new_src)
    print(f"Patch applicata a {operators_path}")
elif "import cupy  # noqa: F401" in src:
    print("Patch gia' presente, salto.")
else:
    raise RuntimeError(f"Blocco da patchare non trovato in {operators_path}.")

Patch applicata a /usr/local/lib/python3.13/dist-packages/IPPy/operators.py


In [4]:
import math
import time
import shutil

import numpy as np
import torch
from tqdm.auto import tqdm

from IPPy import operators
from IPPy.solvers import ChambollePockTpVConstrained
from IPPy.utilities import metrics
from IPPy.nn.models import UNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo in uso: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("Verifica che il runtime sia impostato su L4 in Runtime > Cambia tipo di runtime.")

Dispositivo in uso: cuda
GPU: NVIDIA L4
Verifica che il runtime sia impostato su L4 in Runtime > Cambia tipo di runtime.


In [5]:
# ============================================================
# Parametri e proiettori (stessa geometria usata in tutta la pipeline)
# ============================================================
IMG_SIZE = (256, 256)
NOISE_LEVEL = 0.005
MAX_ITER = 300   # come nella generazione finale di TV; ritarabile in seguito

ANGLE_CONFIGS = {
    90: np.linspace(-45, 45, 90),
    45: np.linspace(-45, 45, 45),
    30: np.linspace(-30, 30, 32)[1:-1],
    15: np.linspace(-30, 30, 17)[1:-1],
}

# Parametri tarati con 06_WeightedTV_hyperparameter_tuning.ipynb (validation,
# 12 campioni, score combinato PSNR+SSIM 50/50) -- non piu' presi in prestito
# da TV/paper, uno per ciascuna configurazione angolare.
LAMBDAS = {90: 0.2, 45: 0.2, 30: 0.03, 15: 0.01}
ETAS = {90: 2e-3, 45: 5e-3, 30: 1e-3, 15: 5e-3}
PS = {90: 0.1, 45: 0.1, 30: 0.5, 15: 0.3}

PROJECTORS = {
    n_angles: operators.CTProjector(
        img_shape=IMG_SIZE,
        angles=np.deg2rad(angles),
        geometry="parallel",
        force_cpu=False,   # su GPU (L4)
    )
    for n_angles, angles in ANGLE_CONFIGS.items()
}

Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA


In [6]:
# ============================================================
# Caricamento dei 4 modelli UNet dai checkpoint salvati su Drive durante
# il training (04_unet.ipynb): /content/drive/MyDrive/unet_checkpoints/<n_angoli>/model_weights.pth
# ============================================================

UNET_CHECKPOINTS_DIR = Path("/content/drive/MyDrive/unet_checkpoints")


def infer_unet_config(state_dict):
    """Ricostruisce i parametri dell'architettura dai pesi salvati.
    'final_activation' non ha pesi propri: va impostata a mano, coerente
    con MODEL_CONFIG usata in 04_unet.ipynb."""
    ch_in = state_dict["preprocess.conv.0.weight"].shape[1]
    middle_ch = [state_dict["preprocess.conv.0.weight"].shape[0]]

    i = 0
    while f"encoder_layers.{i}.conv.conv2.weight" in state_dict:
        middle_ch.append(state_dict[f"encoder_layers.{i}.conv.conv2.weight"].shape[0])
        i += 1
    n_blocks = i

    ch_out = state_dict["postprocess.weight"].shape[0]

    return {
        "ch_in": ch_in,
        "ch_out": ch_out,
        "middle_ch": middle_ch,
        "n_layers_per_block": 2,
        "down_layers": tuple(["ResDownBlock"] * n_blocks),
        "up_layers": tuple(["ResUpBlock"] * n_blocks),
        "final_activation": "sigmoid",
    }


def load_unet_model(n_angles: int) -> torch.nn.Module:
    weights_path = UNET_CHECKPOINTS_DIR / str(n_angles) / "model_weights.pth"
    state_dict = torch.load(weights_path, map_location=DEVICE)

    config = infer_unet_config(state_dict)
    print(f"{n_angles} angoli -> {config}")

    model = UNet(**config)
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    return model


models = {n_angles: load_unet_model(n_angles) for n_angles in ANGLE_CONFIGS}

90 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock'), 'final_activation': 'sigmoid'}
45 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock'), 'final_activation': 'sigmoid'}
30 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock'), 'final_activation': 'sigmoid'}
15 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock'), 'final_activation': 'sigmoid'}


In [7]:
# ============================================================
# Solver Weighted TV: sottoclasse di ChambollePockTpVConstrained (IPPy)
# che calcola i pesi UNA SOLA VOLTA da x_tilde (output UNet), invece di
# ricalcolarli ogni iterazione dalla soluzione corrente xx.
#
# Tutto il resto (power method, update di x/y/w, criteri di stop) e'
# IDENTICO all'originale IPPy -- copiato senza modifiche, cambia solo
# DOVE viene calcolato W (fuori dal ciclo invece che dentro).
#
# NOTA: qui non riapplichiamo il decoratore @on_batch dell'originale:
# lo chiamiamo sempre con batch=1 (un'immagine alla volta, come fate
# gia' in tutta la pipeline), quindi la logica di batching non serve.
# ============================================================

class ChambollePockWeightedTpVConstrained(ChambollePockTpVConstrained):

    def __call__(
        self,
        y_delta,
        epsilon,
        lmbda,
        x_tilde,
        x_true=None,
        starting_point=None,
        eta=2e-3,
        maxiter=100,
        p=0.3,
        verbose=False,
        *args,
        **kwargs,
    ):
        assert x_tilde is not None, "x_tilde (es. output UNet) e' obbligatorio per la Weighted TV."

        nu = math.sqrt(
            self.power_method(self.K, num_iterations=10)
            / self.power_method(self.grad, num_iterations=10)
        )
        Gamma = math.sqrt(
            self.power_method_dual_operator(self.K, self.grad, num_iterations=10)
        )
        tau = 1 / Gamma
        sigma = 1 / Gamma
        theta = 1

        k = 0
        x = torch.zeros((1, 1, self.nx, self.ny), device=y_delta.device) if starting_point is None else starting_point
        y = torch.zeros((1, 1, self.mx, self.my), device=y_delta.device)
        w = torch.zeros((1, 2, self.nx, self.ny), device=y_delta.device)
        xx = x

        # ------------------------------------------------------------
        # QUI la modifica rispetto all'originale: pesi calcolati una
        # volta sola da x_tilde (fisso), non ricalcolati nel ciclo.
        # ------------------------------------------------------------
        grad_x_tilde = self.grad(x_tilde)
        grad_mag_tilde = torch.square(grad_x_tilde[:, 0:1]) + torch.square(grad_x_tilde[:, 1:2])
        W = (torch.sqrt(eta**2 + grad_mag_tilde) / eta) ** (p - 1)
        WW = torch.cat((W, W), dim=1)
        # ------------------------------------------------------------

        info = dict()
        info["residues"] = torch.zeros((maxiter + 1, 1))
        info["obj"] = torch.zeros((maxiter + 1, 1))
        info["RE"] = torch.zeros((maxiter + 1, 1))
        info["RMSE"] = torch.zeros((maxiter + 1, 1))
        info["PSNR"] = torch.zeros((maxiter + 1, 1))
        info["SSIM"] = torch.zeros((maxiter + 1, 1))
        info["iterations"] = 0

        con = True
        while con and (k < maxiter):
            yy = y + sigma * (self.K(xx) - y_delta)
            y = max(torch.norm(yy) - (sigma * epsilon), 0) * yy / torch.norm(yy)

            # (nessun ricalcolo di W qui: e' fisso, calcolato sopra da x_tilde)

            x_grad = self.grad(xx)
            ww = w + sigma * nu * x_grad
            abs_ww = torch.square(ww[:, 0:1]) + torch.square(ww[:, 1:2])
            abs_ww = torch.cat((abs_ww, abs_ww), dim=1)

            lmbda_vec_over_nu = lmbda * WW / nu
            w = lmbda_vec_over_nu * ww / torch.maximum(lmbda_vec_over_nu, abs_ww)

            xtmp = x
            x = xtmp - tau * (self.K.T(y) + nu * self.grad.T(w))
            x[x < 0] = 0
            xx = x + theta * (x - xtmp)

            if x_true is not None:
                info["RE"][k] = metrics.RE(x, x_true)
                info["PSNR"][k] = metrics.PSNR(x, x_true)
                info["RMSE"][k] = metrics.RMSE(x, x_true)
                info["SSIM"][k] = metrics.SSIM(x, x_true)

            grad_x = self.grad(x)
            grad_mag = torch.sqrt(torch.square(grad_x[:, 0:1]) + torch.square(grad_x[:, 1:2]))
            ftpv = torch.sum(torch.abs(W * grad_mag))
            res = torch.norm(self.K(x) - y_delta, 2) ** 2

            info["residues"][k] = res
            info["obj"][k] = 0.5 * res + lmbda * ftpv

            c = math.sqrt(res) / (torch.max(y_delta) * math.sqrt(self.mx * self.my))
            d_abs = torch.norm(x.flatten() - xtmp.flatten())

            if (c >= 9e-6) and (c <= 1.1e-5):
                con = False
            if d_abs < 1e-3 * (1 + torch.norm(xtmp.flatten())):
                con = False

            k = k + 1
            if verbose and k % 50 == 0:
                print(f"Iterazione {k}/{maxiter}")

        info["residues"] = info["residues"][:k]
        info["obj"] = info["obj"][:k]
        info["RE"] = info["RE"][:k]
        info["RMSE"] = info["RMSE"][:k]
        info["PSNR"] = info["PSNR"][:k]
        info["SSIM"] = info["SSIM"][:k]
        info["iterations"] = k
        return x, info

In [8]:
def quick_test():
    test_n_angles = 90
    sino_dir = LOCAL_SINOGRAMS_DIR / "test" / str(test_n_angles)
    fbp_dir = LOCAL_FBP_DIR / "test" / str(test_n_angles)
    sino_paths = sorted(sino_dir.rglob("*.npy"))
    if not sino_paths:
        print("Nessun sinogramma di test trovato, salto il test rapido.")
        return

    sino_path = sino_paths[0]
    rel_path = sino_path.relative_to(sino_dir)
    fbp_path = fbp_dir / rel_path

    solver = ChambollePockWeightedTpVConstrained(PROJECTORS[test_n_angles])
    model = models[test_n_angles]

    sinogram = np.load(sino_path)
    y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)   # CPU: CTProjector di IPPy non ha il path GPU implementato
    epsilon = NOISE_LEVEL * torch.norm(y_delta)
    starting_point = torch.zeros((1, 1, *IMG_SIZE))   # CPU

    fbp = np.clip(np.load(fbp_path).astype(np.float32), 0.0, 1.0)
    fbp_t = torch.from_numpy(fbp).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        x_tilde = model(fbp_t).cpu()   # UNet su GPU, risultato riportato su CPU prima del solver

    t0 = time.time()
    with torch.no_grad():
        x_sol, info = solver(
            y_delta, epsilon=epsilon, lmbda=LAMBDAS[test_n_angles], x_tilde=x_tilde,
            x_true=None, starting_point=starting_point,
            eta=ETAS[test_n_angles], maxiter=MAX_ITER, p=PS[test_n_angles], verbose=False,
        )
    elapsed = time.time() - t0

    n_totale = sum(len(list((LOCAL_SINOGRAMS_DIR / "test" / str(n)).rglob("*.npy"))) for n in ANGLE_CONFIGS)
    print(f"Tempo per una immagine ({test_n_angles} angoli, {MAX_ITER} iter max): {elapsed:.2f} s")
    print(f"Iterazioni effettive: {info['iterations']} / {MAX_ITER}")
    print(f"Stima per l'intero test set ({n_totale} immagini totali, 4 config): {elapsed * n_totale / 60:.1f} minuti")

quick_test()

Tempo per una immagine (90 angoli, 300 iter max): 1.61 s
Iterazioni effettive: 300 / 300
Stima per l'intero test set (1308 immagini totali, 4 config): 35.0 minuti


In [9]:
# ============================================================
# Utility di sincronizzazione incrementale su Drive
# ============================================================
def sync_to_drive(src_dir: Path, dst_dir: Path):
    if not src_dir.exists():
        return
    dst_dir.mkdir(parents=True, exist_ok=True)
    n_copied = 0
    for f in src_dir.rglob("*.npy"):
        rel = f.relative_to(src_dir)
        dst = dst_dir / rel
        if dst.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
        n_copied += 1
    if n_copied:
        print(f"Sincronizzati {n_copied} nuovi file su Drive ({dst_dir}).")

In [10]:
SYNC_EVERY = 100
SPLIT = "test"

LOCAL_OUTPUT_DIR = Path("/content/weighted_tv_reconstructions_tuned")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/weighted_tv_reconstructions_tuned")

failures = []

for n_angles, K in PROJECTORS.items():
    print(f"\n--> Angoli: {n_angles} | lambda: {LAMBDAS[n_angles]} | eta: {ETAS[n_angles]} | p: {PS[n_angles]}")

    solver = ChambollePockWeightedTpVConstrained(K)
    model = models[n_angles]
    lmbda = LAMBDAS[n_angles]

    input_sino_dir = LOCAL_SINOGRAMS_DIR / SPLIT / str(n_angles)
    input_fbp_dir = LOCAL_FBP_DIR / SPLIT / str(n_angles)
    local_out_dir = LOCAL_OUTPUT_DIR / SPLIT / str(n_angles)
    drive_out_dir = DRIVE_OUTPUT_DIR / SPLIT / str(n_angles)

    sino_paths = sorted(
        input_sino_dir.rglob("*.npy"),
        key=lambda p: (p.parent.name, int(p.stem))
    )

    n_since_sync = 0

    for sino_path in tqdm(sino_paths, desc=f"Weighted TV [{SPLIT}-{n_angles} deg]"):
        rel_path = sino_path.relative_to(input_sino_dir)
        local_path = local_out_dir / rel_path
        drive_path = drive_out_dir / rel_path

        if drive_path.exists():
            continue

        local_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            sinogram = np.load(sino_path)
            y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)   # CPU
            epsilon = NOISE_LEVEL * torch.norm(y_delta)
            starting_point = torch.zeros((1, 1, *IMG_SIZE))   # CPU

            fbp_path = input_fbp_dir / rel_path
            fbp = np.clip(np.load(fbp_path).astype(np.float32), 0.0, 1.0)
            fbp_t = torch.from_numpy(fbp).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                x_tilde = model(fbp_t).cpu()   # UNet su GPU, poi torna su CPU

            with torch.no_grad():
                x_sol, _ = solver(
                    y_delta, epsilon=epsilon, lmbda=lmbda, x_tilde=x_tilde,
                    x_true=None, starting_point=starting_point,
                    eta=ETAS[n_angles], maxiter=MAX_ITER, p=PS[n_angles], verbose=False,
                )

            np.save(local_path, x_sol.squeeze().cpu().numpy().astype(np.float32))

        except Exception as e:
            failures.append((str(sino_path), str(e)))
            continue

        n_since_sync += 1
        if n_since_sync >= SYNC_EVERY:
            sync_to_drive(local_out_dir, drive_out_dir)
            n_since_sync = 0

    sync_to_drive(local_out_dir, drive_out_dir)

print(f"\nFallimenti totali: {len(failures)}")
for path, err in failures[:5]:
    print(path)
    print(" ->", err)


--> Angoli: 90 | lambda: 0.2 | eta: 0.002 | p: 0.1


Weighted TV [test-90 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/90).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/90).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/90).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/90).

--> Angoli: 45 | lambda: 0.2 | eta: 0.005 | p: 0.1


Weighted TV [test-45 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/45).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/45).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/45).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/45).

--> Angoli: 30 | lambda: 0.03 | eta: 0.001 | p: 0.5


Weighted TV [test-30 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/30).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/30).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/30).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/30).

--> Angoli: 15 | lambda: 0.01 | eta: 0.005 | p: 0.3


Weighted TV [test-15 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/15).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/15).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/15).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/weighted_tv_reconstructions_tuned/test/15).

Fallimenti totali: 0
